In [1]:
!pip install rdkit -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 16.4 MB/s eta 0:00:00


In [3]:
# Standard library
import os
import re
import sys
from collections import Counter, defaultdict

# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Draw, rdChemReactions, rdDepictor
from rdkit.Chem.Draw import IPythonConsole, rdMolDraw2D
from rdkit.Chem.Draw.MolDrawing import DrawingOptions, MolDrawing

# Local imports
from template_extractor import extract_from_reaction

# RDKit settings
RDLogger.DisableLog("rdApp.*")
rdDepictor.SetPreferCoordGen(True)

# Example usage
os.getcwd()

'/content'

In [4]:
def build_template_extractor(args):
    setting = {'verbose': False, 'use_stereo': False, 'use_symbol': False, 'max_unmap': 5, 'retro': False, 'remote': True, 'least_atom_num': 0}
    for k in setting.keys():
        if k in args.keys():
            setting[k] = args[k]
    print ('Template extractor setting:', setting)
    return lambda x: extract_from_reaction(x, setting)

def get_reaction_template(extractor, rxn, _id = 0):
    rxn = {'reactants': rxn.split('>>')[0], 'products': rxn.split('>>')[1], '_id': _id}
    result = extractor(rxn)
    return rxn, result

def get_full_template(template, H_change, Charge_change, Chiral_change):
    H_code = ''.join([str(H_change[k+1]) for k in range(len(H_change))])
    Charge_code = ''.join([str(Charge_change[k+1]) for k in range(len(Charge_change))])
    Chiral_code = ''.join([str(Chiral_change[k+1]) for k in range(len(Chiral_change))])
    return '_'.join([template, H_code, Charge_code, Chiral_code])

In [5]:
args ={'verbose': False, 'use_stereo': False, 'use_symbol': True, 'max_unmap': 5, 'retro': False, 'remote': True, 'least_atom_num': 0,
      'dataset':'USPTO_Mechanism'}

In [6]:
extractor = build_template_extractor(args)

Template extractor setting: {'verbose': False, 'use_stereo': False, 'use_symbol': True, 'max_unmap': 5, 'retro': False, 'remote': True, 'least_atom_num': 0}


In [7]:
#Example atom mapped mechanism
atom_mapped_rxns_list = ['[Cl:1][c:2]1[cH:3][cH:4][cH:5][cH:6][cH:7]1.[CH3:8][CH2:9][NH2:10].[Pd:11][P+:12]([C:19]1=[CH:20][CH:21]=[CH:22][CH:23]=[CH:24]1)([C:25]1=[CH:26][CH:27]=[CH:28][CH:29]=[CH:30]1)[c:13]1[cH:14][cH:15][cH:16][cH:17][cH:18]1.[CH3:31][C:32]([CH3:33])([CH3:34])[O-:35].[Na+:36]>>[CH3:8][CH2:9][NH2:10].[CH3:31][C:32]([CH3:33])([CH3:34])[O-:35].[Cl:1][Pd:11]([C:2]1=[CH:3][CH:4]=[CH:5][CH:6]=[CH:7]1)[P+:12]([C:13]1=[CH:14][CH:15]=[CH:16][CH:17]=[CH:18]1)([C:25]1=[CH:26][CH:27]=[CH:28][CH:29]=[CH:30]1)[c:19]1[cH:20][cH:21][cH:22][cH:23][cH:24]1.[Na+:36]',
                         '[CH3:1][CH2:2][NH2:3].[CH3:4][C:5]([CH3:6])([CH3:7])[O-:8].[Cl:9][Pd:10]([C:30]1=[CH:31][CH:32]=[CH:33][CH:34]=[CH:35]1)[P+:11]([C:18]1=[CH:19][CH:20]=[CH:21][CH:22]=[CH:23]1)([C:24]1=[CH:25][CH:26]=[CH:27][CH:28]=[CH:29]1)[c:12]1[cH:13][cH:14][cH:15][cH:16][cH:17]1.[Na+:36]>>[CH3:4][C:5]([CH3:6])([CH3:7])[O-:8].[CH3:1][CH2:2][NH2+:3][Pd:10]([C:30]1=[CH:31][CH:32]=[CH:33][CH:34]=[CH:35]1)[P+:11]([C:18]1=[CH:19][CH:20]=[CH:21][CH:22]=[CH:23]1)([C:24]1=[CH:25][CH:26]=[CH:27][CH:28]=[CH:29]1)[c:12]1[cH:13][cH:14][cH:15][cH:16][cH:17]1.[Na+:36].[Cl-:9]',
                         '[CH3:1][C:2]([CH3:3])([CH3:4])[O-:5].[CH3:6][CH2:7][NH2+:8][Pd:9]([C:29]1=[CH:30][CH:31]=[CH:32][CH:33]=[CH:34]1)[P+:10]([C:17]1=[CH:18][CH:19]=[CH:20][CH:21]=[CH:22]1)([C:23]1=[CH:24][CH:25]=[CH:26][CH:27]=[CH:28]1)[c:11]1[cH:12][cH:13][cH:14][cH:15][cH:16]1.[Na+:35].[Cl-:36]>>[CH3:1][C:2]([CH3:3])([CH3:4])[OH:5].[CH3:6][CH2:7][NH:8][Pd:9]([C:29]1=[CH:30][CH:31]=[CH:32][CH:33]=[CH:34]1)[P+:10]([C:17]1=[CH:18][CH:19]=[CH:20][CH:21]=[CH:22]1)([C:23]1=[CH:24][CH:25]=[CH:26][CH:27]=[CH:28]1)[c:11]1[cH:12][cH:13][cH:14][cH:15][cH:16]1.[Na+:35].[Cl-:36]',
                         '[CH3:1][C:2]([CH3:3])([CH3:5])[OH:4].[CH3:6][CH2:7][NH:8][Pd:9]([C:10]1=[CH:11][CH:12]=[CH:13][CH:14]=[CH:15]1)[P+:16]([C:17]1=[CH:18][CH:19]=[CH:20][CH:21]=[CH:22]1)([C:23]1=[CH:24][CH:25]=[CH:26][CH:27]=[CH:28]1)[c:29]1[cH:30][cH:31][cH:32][cH:33][cH:34]1.[Na+:35].[Cl-:36]>>[CH3:1][C:2]([CH3:3])([CH3:5])[OH:4].[Pd:9][P+:16]([C:17]1=[CH:18][CH:19]=[CH:20][CH:21]=[CH:22]1)([C:23]1=[CH:24][CH:25]=[CH:26][CH:27]=[CH:28]1)[c:29]1[cH:30][cH:31][cH:32][cH:33][cH:34]1.[CH3:6][CH2:7][NH:8][C:10]1=[CH:11][CH:12]=[CH:13][CH:14]=[CH:15]1.[Na+:35].[Cl-:36]',
                         ]

In [8]:
templates = []
for rxn in atom_mapped_rxns_list:
  output = get_reaction_template(extractor, rxn, _id = 0)
  templates.append((output[1]['reaction_smarts'], output[1]['intra_only']))
templates

[('[Pd;H0;D1;+0:11].[Cl;H0;D1;+0:1]-[c;H0;D3;+0:2]>>[Cl;H0;D1;+0:1]-[Pd;H0;D3;+0:11]-[c;H0;D3;+0:2]',
  False),
 ('[NH2;D1;+0:3].[Cl;H0;D1;+0:9]-[Pd;H0;D3;+0:10]>>[Cl-;H0;D0:9].[NH2+;D2:3]-[Pd;H0;D3;+0:10]',
  False),
 ('[NH2+;D2:8].[O-;H0;D1:5]>>[OH;D1;+0:5].[NH;D2;+0:8]', False),
 ('[NH;D2;+0:8]-[Pd;H0;D3;+0:9]-[c;H0;D3;+0:10]>>[Pd;H0;D1;+0:9].[NH;D2;+0:8]-[c;H0;D3;+0:10]',
  True)]

In [9]:
templates_ = []
for i in templates:
  r, p = i[0].split('>>')
  templates_.append(((f"({r})>>{p}"), i[1]))
templates_

[('([Pd;H0;D1;+0:11].[Cl;H0;D1;+0:1]-[c;H0;D3;+0:2])>>[Cl;H0;D1;+0:1]-[Pd;H0;D3;+0:11]-[c;H0;D3;+0:2]',
  False),
 ('([NH2;D1;+0:3].[Cl;H0;D1;+0:9]-[Pd;H0;D3;+0:10])>>[Cl-;H0;D0:9].[NH2+;D2:3]-[Pd;H0;D3;+0:10]',
  False),
 ('([NH2+;D2:8].[O-;H0;D1:5])>>[OH;D1;+0:5].[NH;D2;+0:8]', False),
 ('([NH;D2;+0:8]-[Pd;H0;D3;+0:9]-[c;H0;D3;+0:10])>>[Pd;H0;D1;+0:9].[NH;D2;+0:8]-[c;H0;D3;+0:10]',
  True)]

In [10]:
from itertools import permutations
from rdkit import Chem
from collections import Counter


def out_(outcome):
  mapnums = [a.GetAtomMapNum() for m in outcome for a in m.GetAtoms() if a.GetAtomMapNum()]
  #print(mapnums)
  if len(mapnums) != len(set(mapnums)): # duplicate?

      merged_mol = Chem.RWMol(outcome[0])
      merged_map_to_id = {a.GetAtomMapNum(): a.GetIdx() for a in outcome[0].GetAtoms() if a.GetAtomMapNum()}
      for j in range(1, len(outcome)):
          new_mol = outcome[j]
          for a in new_mol.GetAtoms():
              if a.GetAtomMapNum() not in merged_map_to_id:
                  merged_map_to_id[a.GetAtomMapNum()] = merged_mol.AddAtom(a)
          for b in new_mol.GetBonds():
              bi = b.GetBeginAtom().GetAtomMapNum()
              bj = b.GetEndAtom().GetAtomMapNum()

              if not merged_mol.GetBondBetweenAtoms(
                      merged_map_to_id[bi], merged_map_to_id[bj]):
                  merged_mol.AddBond(merged_map_to_id[bi],
                      merged_map_to_id[bj], b.GetBondType())
                  merged_mol.GetBondBetweenAtoms(
                      merged_map_to_id[bi], merged_map_to_id[bj]
                  ).SetStereo(b.GetStereo())
                  merged_mol.GetBondBetweenAtoms(
                      merged_map_to_id[bi], merged_map_to_id[bj]
                  ).SetBondDir(b.GetBondDir())
      outcome = merged_mol.GetMol()

  else:
      new_outcome = outcome[0]
      for j in range(1, len(outcome)):
          new_outcome = AllChem.CombineMols(new_outcome, outcome[j])
      outcome = new_outcome

  return outcome


#this code is from:
def make_rxns(source_rxn, reactants, intra_only=False):
    new_rxns = []
    product_sets = source_rxn.RunReactants(reactants)
    #print('product_sets before:\n', product_sets, '\n')

    if intra_only:
      product_sets = [(out_(pset),) for pset in product_sets]

    for pset in product_sets:
        new_rxn = AllChem.ChemicalReaction()
        for react in reactants:
            react = Chem.Mol(react)
            for a in react.GetAtoms():
                a.SetIntProp('molAtomMapNumber', a.GetIdx())
            new_rxn.AddReactantTemplate(react)
        for prod in pset:
            for a in prod.GetAtoms():
                a.SetIntProp('molAtomMapNumber', int(a.GetProp('react_atom_idx')))
            new_rxn.AddProductTemplate(prod)
            #print('new reaction:===>\n', AllChem.ReactionToSmiles(new_rxn))

            #----some tweaking, needed for one edge case---------
            # Get the reaction SMILES
            rxn_smiles = AllChem.ReactionToSmiles(new_rxn)

            # Tweak: Remove parentheses if they enclose the product part
            reactant_smiles, product_smiles = rxn_smiles.split('>>')
            if product_smiles.startswith('(') and product_smiles.endswith(')'):
                product_smiles = product_smiles[1:-1]
            tweaked_rxn_smiles = f"{reactant_smiles}>>{product_smiles}"

            # Convert back to a `ChemicalReaction` object
            tweaked_rxn = AllChem.ReactionFromSmarts(tweaked_rxn_smiles)
            new_rxns.append(tweaked_rxn)
            #----------------------------------------------------------
    return new_rxns

def match_pdts(true_pdts, gen_pdts):
    # Split each input string into separate SMILES components
    true_parts = true_pdts.split('.')
    gen_parts = gen_pdts.split('.')
    # Canonicalize each component of true_pdts
    true_canonicals = set()
    for smi in true_parts:
        mol = Chem.MolFromSmiles(smi)
        [a.SetAtomMapNum(0) for a in mol.GetAtoms()]
        if mol is not None:
            # Convert to canonical SMILES without stereochemistry
            true_canonicals.add(Chem.MolToSmiles(mol, isomericSmiles=False))

    # Canonicalize and compare each component of gen_pdts
    for smi in gen_parts:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            [a.SetAtomMapNum(0) for a in mol.GetAtoms()]
            gen_canonical = Chem.MolToSmiles(mol, isomericSmiles=False)
            if gen_canonical in true_canonicals:
                return True

    return False

def find_all_reac_reag_comb(template_smarts_str, str_of_smiles, intra_only=False):
    #print('these are getting used:', template_smarts_str, str_of_smiles)
    list_of_smiles = str_of_smiles.split('.')
    #print('all smiles:', list_of_smiles)
    reactants_smarts = template_smarts_str.split('>>')[0][1:-1]
    reactants_smarts = reactants_smarts.split('.')
    #print('reactants_smarts:', reactants_smarts)
    # Dictionary to hold matches for each SMARTS pattern
    smarts_to_matches = {smarts: [] for smarts in reactants_smarts}

    if intra_only:
        # Check if a single molecule contains all SMARTS
        results = []
        for smiles in list_of_smiles:
            mol = Chem.MolFromSmiles(smiles)
            if all(mol.HasSubstructMatch(Chem.MolFromSmarts(smarts)) for smarts in reactants_smarts):
                # Intramolecular case: all SMARTS must match within the same molecule
                reagents = list_of_smiles.copy()
                reagents.remove(smiles)  # Remaining molecules are considered reagents
                results.append(([smiles], reagents))
        return results

    for smiles in list_of_smiles:
        mol = Chem.MolFromSmiles(smiles)
        for smarts in reactants_smarts:
            #print('smarts here:\n', smarts)
            smarts_mol = Chem.MolFromSmarts(smarts)
            if mol.HasSubstructMatch(smarts_mol):
                smarts_to_matches[smarts].append(smiles)

    # Generate all permutations of matches for each SMARTS pattern
    reactant_combinations = []
    for reactant_permutation in permutations(list_of_smiles, len(reactants_smarts)):
        match = True
        for reactant, smarts in zip(reactant_permutation, reactants_smarts):
            mol = Chem.MolFromSmiles(reactant)
            smarts_mol = Chem.MolFromSmarts(smarts)
            if not mol.HasSubstructMatch(smarts_mol):
                match = False
                break
        if match:
            reactant_combinations.append(list(reactant_permutation))


    # Remove duplicate combinations
    unique_combinations = []
    for combo in reactant_combinations:
        if combo not in unique_combinations:
            unique_combinations.append(combo)

    # Generate reagents for each reactant combination, maintaining counts
    results = []
    original_counts = Counter(list_of_smiles)  # Keep track of the original counts
    for reactants in unique_combinations:
        reactant_counts = Counter(reactants)
        reagents_counts = original_counts - reactant_counts  # Subtract reactant counts
        reagents = list(reagents_counts.elements())  # Reconstruct the reagents list
        results.append((reactants, reagents))


    return results


def transform_func(template_smarts_str_, str_of_smiles):
    template_smarts_str = template_smarts_str_[0]
    intra_only = template_smarts_str_[1]

    reactant_reagents = find_all_reac_reag_comb(template_smarts_str, str_of_smiles, intra_only)
    #print('reactant_reagents:-->\n', reactant_reagents)
    # if not reactant_reagents:
    #     print(f"No valid reactant-reagent combinations found for: {template_smarts_str} with {str_of_smiles}")
    #     return []  # Return an empty list or handle appropriately


    all_mapped_rxns = []  # Store results for all reactant-reagent combinations

    for reactants_smiles_list, reagents_smiles_list in reactant_reagents:
        reagent_smiles = '.'.join(reagents_smiles_list)
        reactants_mols_list = [Chem.MolFromSmiles(smiles) for smiles in reactants_smiles_list]
        template = AllChem.ReactionFromSmarts(template_smarts_str)

        # -------------------combine the reactants into a single mol object--------------
        rmol = None
        for mol in reactants_mols_list:
            if rmol is None:
                rmol = mol  # If rmol is None, set it to the first molecule
            else:
                rmol = Chem.CombineMols(rmol, mol)

        # ----------------------mapping the reactants and products-------------------------
        atom_mapped_rxns = []
        for r in make_rxns(template, [rmol], intra_only):
            #print('template, intra_only:\n', template, intra_only, '\n')
            smi = AllChem.ReactionToSmiles(r)
            #print('atom_mapped_rxns before cleaning:', smi, '\n')
            smi = re.sub(r'^\((.*)\)>', r'\1>', smi)  # Clean up the reaction SMARTS
            atom_mapped_rxns.append(smi)
            #print('atom_mapped_rxns after cleaning:', atom_mapped_rxns, '\n')
        # --------------------------adding the reagents------------------------------------
        for i in atom_mapped_rxns:
            mapped_rxn = i.replace(">>", f">{reagent_smiles}>")
            all_mapped_rxns.append(mapped_rxn)

    # Remove duplicates from all mapped reactions
    all_mapped_rxns = list(set(all_mapped_rxns))
    return all_mapped_rxns


def dfs_with_processing(start, transform_func, validate_func, target_output, param_list):
    visited = set()  # To track visited states and avoid cycles

    def process_output(output):
        """Custom processing logic on the outputs if required."""
        try:
            output_reagent = output.split('>')[1]
            output_pdt = output.split('>')[2]
            if output_reagent:
                processed = '.'.join([output_pdt, output_reagent])
            else:
                processed = output_pdt
            return processed

        except IndexError:
            return None  # If processing fails, return None to signal an invalid path

    def is_valid_molecule(smiles):
        """Check if a SMILES string corresponds to a valid molecule."""
        try:
            mol = Chem.MolFromSmiles(smiles)
            return mol is not None
        except:
            return False

    def dfs(current_input, path, param_index):
        # Stop if the current input matches the target output
        if validate_func(current_input, target_output):
            #print('Got it, matched!')
            return path

        if (current_input, param_index) in visited:
            return None
        visited.add((current_input, param_index))

        # Check if we've exhausted all transformation rules
        if param_index >= len(param_list):
            return None

        # Apply the current transformation rule
        current_param = param_list[param_index]
        raw_outputs = transform_func(current_param, current_input)  # Raw outputs from transform_func
        #print('raw_outputs:\n', raw_outputs)
        processed_outputs = []

        for raw in raw_outputs:
            # Process the raw output
            processed = process_output(raw)
            if processed is None:
                continue  # Skip invalid processing results

            # Validate individual SMILES in the processed output
            components = processed.split('.')
            if all(is_valid_molecule(comp) for comp in components):
                processed_outputs.append((raw, processed))
        #print('processed_outputs=======>\n', processed_outputs)
        for raw_output, processed_output in processed_outputs:
            # Recursive DFS with the next SMARTS pattern in param_list
            result = dfs(processed_output, path + [(current_param, raw_output, processed_output)], param_index + 1)
            if result is not None:
                return result

        return None  # No valid path found

    # Start with the first SMARTS pattern
    return dfs(start, [], 0)


In [11]:
#test with one example
start = 'Clc1ccccc1.CCN.[Pd][P+](c2ccccc2)(C3=CC=CC=C3)C4=CC=CC=C4.CC(C)(C)[O-].[Na+]'
target_output = 'CCNC1=CC=CC=C1'
results = dfs_with_processing(start, transform_func, match_pdts, target_output, templates_)
[results[i][1] for i in range(len(results))]

['[Cl:20][c:21]1[cH:22][cH:23][cH:24][cH:25][cH:26]1.[Pd:0][P+:1]([c:2]1[cH:3][cH:4][cH:5][cH:6][cH:7]1)([c:8]1[cH:9][cH:10][cH:11][cH:12][cH:13]1)[c:14]1[cH:15][cH:16][cH:17][cH:18][cH:19]1>CCN.CC(C)(C)[O-].[Na+]>[Pd:0]([P+:1]([c:2]1[cH:3][cH:4][cH:5][cH:6][cH:7]1)([c:8]1[cH:9][cH:10][cH:11][cH:12][cH:13]1)[c:14]1[cH:15][cH:16][cH:17][cH:18][cH:19]1)([Cl:20])[c:21]1[cH:22][cH:23][cH:24][cH:25][cH:26]1',
 '[CH3:0][CH2:1][NH2:2].[Pd:3]([P+:4]([c:5]1[cH:6][cH:7][cH:8][cH:9][cH:10]1)([c:11]1[cH:12][cH:13][cH:14][cH:15][cH:16]1)[c:17]1[cH:18][cH:19][cH:20][cH:21][cH:22]1)([Cl:23])[c:24]1[cH:25][cH:26][cH:27][cH:28][cH:29]1>CC(C)(C)[O-].[Na+]>[CH3:0][CH2:1][NH2+:2][Pd:3]([P+:4]([c:5]1[cH:6][cH:7][cH:8][cH:9][cH:10]1)([c:11]1[cH:12][cH:13][cH:14][cH:15][cH:16]1)[c:17]1[cH:18][cH:19][cH:20][cH:21][cH:22]1)[c:24]1[cH:25][cH:26][cH:27][cH:28][cH:29]1.[Cl-:23]',
 '[CH3:0][CH2:1][NH2+:2][Pd:3]([P+:4]([c:5]1[cH:6][cH:7][cH:8][cH:9][cH:10]1)([c:11]1[cH:12][cH:13][cH:14][cH:15][cH:16]1)[c:17]1[cH:18

In [12]:
#apply to all the reactions

In [13]:
def get_mechanism(start, target):
  results = dfs_with_processing(start, transform_func, match_pdts, target, templates_)
  if results:
    return [results[i][1] for i in range(len(results))]
  else:
    return None

In [14]:
df = pd.read_csv('/content/bha_reactants_products.csv')
df.head(2)

,reactants,product
0,Cc1ccccc1Cl.CCCCN.CN(C)c1ccc([P+]([Pd])([C@]23...,CCCCNc1ccccc1C
1,Cc1ccccc1Cl.NC1CCCCC1.CN(C)c1ccc([P+]([Pd])([C...,Cc1ccccc1NC1CCCCC1


In [15]:
mechanisms = []
for i, j in zip(df['reactants'], df['product']):
  try:
    mechanisms.append(get_mechanism(i, j))
  except:
    mechanisms.append(None)

len(mechanisms)

314

In [16]:
df['mechanisms'] = mechanisms

In [17]:
df['mechanisms'].iloc[0]

['[CH3:0][N:1]([CH3:2])[c:3]1[cH:4][cH:5][c:6]([P+:7]([Pd:8])([C@:9]23[CH2:10][C@H:11]4[CH2:12][C@H:13]([CH2:14][C@H:15]([CH2:16]4)[CH2:17]2)[CH2:18]3)[C@:19]23[CH2:20][C@H:21]4[CH2:22][C@H:23]([CH2:24][C@H:25]([CH2:26]4)[CH2:27]2)[CH2:28]3)[cH:29][cH:30]1.[CH3:31][c:32]1[cH:33][cH:34][cH:35][cH:36][c:37]1[Cl:38]>CCCCN.CC(C)(C)[O-].[Na+]>[CH3:0][N:1]([CH3:2])[c:3]1[cH:4][cH:5][c:6]([P+:7]([Pd:8]([c:37]2[c:32]([CH3:31])[cH:33][cH:34][cH:35][cH:36]2)[Cl:38])([C@:9]23[CH2:10][C@H:11]4[CH2:12][C@H:13]([CH2:14][C@H:15]([CH2:16]4)[CH2:17]2)[CH2:18]3)[C@:19]23[CH2:20][C@H:21]4[CH2:22][C@H:23]([CH2:24][C@H:25]([CH2:26]4)[CH2:27]2)[CH2:28]3)[cH:29][cH:30]1',
 '[CH3:0][CH2:1][CH2:2][CH2:3][NH2:4].[CH3:5][N:6]([CH3:7])[c:8]1[cH:9][cH:10][c:11]([P+:12]([Pd:13]([c:14]2[c:15]([CH3:16])[cH:17][cH:18][cH:19][cH:20]2)[Cl:21])([C@:22]23[CH2:23][C@H:24]4[CH2:25][C@H:26]([CH2:27][C@H:28]([CH2:29]4)[CH2:30]2)[CH2:31]3)[C@:32]23[CH2:33][C@H:34]4[CH2:35][C@H:36]([CH2:37][C@H:38]([CH2:39]4)[CH2:40]2)[CH2:41]3